# **Prova Estrazione Highlights**

In questa prova utilizzo il dataset di HaSpeeDe, Evalita 2018, la cui definizione usata è:

”HS can be defined as any expression that is abusive, insulting,
intimidating, harassing, and/or incites to violence, hatred, or dis-
crimination. It is directed against people on the basis of their race,
ethnic origin, religion, gender, age, physical condition, disability,
sexual orientation, political conviction, and so forth”

 Abuse (aggression), Age (demographics), Appearance (other characteristics), Disability (demographics), Disease (other characteristics), Ethnicity (demographics), Gender (demographics), Hate (aggression), Insult (verbal/written expressions), Political Opinion (beliefs), Race (demographics), Religion (beliefs), Sexual Orientation (demographics), Violate (aggression), discrimination (discrimination/prejudice), harass (aggression), harassing (aggression), insulting (verbal/written expressions), intimidating (sociocultural control), violence (aggression, physical action).

In particolare utilizzo il dataset preso da twitter.

Il modello che utilizzo per le estrazioni è Mistral-7B-Instruct-v0.3

# HF login e imports

In [1]:
!pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 19.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0


In [ ]:
!hf auth login --token hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `NLP2` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `NLP2`


In [3]:
!pip install --upgrade transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import numpy as np
import json
import random
from IPython.display import display, HTML

# Data Processing

In [5]:
!gdown 13QEcKhGmRxAAx9h_4TeouTpMCV3yA8iO

Downloading...
From: https://drive.google.com/uc?id=13QEcKhGmRxAAx9h_4TeouTpMCV3yA8iO
To: /content/sampled_rows_evalita.csv
100% 13.2k/13.2k [00:00<00:00, 31.2MB/s]


In [10]:
df=pd.read_csv('/content/sampled_rows_evalita.csv', header=None)
df.head()

,0,1
0,273,"@Corriere in Italia, prima gli Italiani Dare l..."
1,2710,Quei migranti che arrivano in palestra al Luzz...
2,2353,Ma tanto prima i profughi! http://t.co/a08ITQg...
3,1661,"\""@LocoPalazzo: @matteosalvinimi stop immigrat..."
4,890,Nessuno che dice nulla su questo #mussulmano h...


In [12]:
texts = df[1]
texts

,1
0,"@Corriere in Italia, prima gli Italiani Dare l..."
1,Quei migranti che arrivano in palestra al Luzz...
2,Ma tanto prima i profughi! http://t.co/a08ITQg...
3,"\""@LocoPalazzo: @matteosalvinimi stop immigrat..."
4,Nessuno che dice nulla su questo #mussulmano h...
...,...
95,Via gli stranieri che non lavorano ...siamo l ...
96,Dopo #Berlino già siamo a #Istanbul. Il #terro...
97,"@agorarai Gerardo, ci sono 100 milioni profugh..."
98,Ancora 5 anni di questo ricambio etnico e l'i...


# Load Model

In [13]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_name="mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config
  )
#6 minuti

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

# Prompt

In [14]:
prompt = [
    {
        "role": "system",
        "content": "You are a data annotator specialized in hate speech detection."
    },
    {
        "role": "user",
        "content": """Read the provided TEXT and extract the exact substring that correspond to the following Hate Speech components.
        If a component is not present in the text, you must return the string "absent".

        Below you find the components to identify in the given text:

        - Abuse
        - Age
        - Appearence
        - Disability
        - Disease
        - Ethnicity
        - Gender
        - Hate
        - Insult
        - Political Opinion
        - Race
        - Discrimination
        - Religion
        - Sexual Orientation
        - Violence
        - Harrassing
        - Insulting
        - Intimidating


        RULES:
        1- You must extract exact quotes from the text. Do not paraphrase or change any words.
        2- Do not output any conversational text, greetings, introductions, or explanations.
        3- Your response must be only the raw json object. Do not wrap the json in markdown formatting.

        OUTPUT FORMAT:
        {
            "Abuse": "extracted substring or absent",
            "Age": "extracted substring or absent",
            "Appearence": "extracted substring or absent",
            "Disability": "extracted substring or absent",
            "Disease": "extracted substring or absent",
            "Ethnicity": "extracted substring or absent",
            "Gender": "extracted substring or absent",
            "Hate": "extracted substring or absent",
            "Insult": "extracted substring or absent",
            "Political Opinion": "extracted substring or absent",
            "Race": "extracted substring or absent",
            "Discrimination": "extracted substring or absent",
            "Religion": "extracted substring or absent",
            "Sexual Orientation": "extracted substring or absent",
            "Violence": "extracted substring or absent",
            "Harrassing": "extracted substring or absent",
            "Insulting": "extracted substring or absent",
            "Intimidating": "extracted substring or absent"
        }


        TEXT: "{text}"

        ANSWER:

        """
    }
]

In [ ]:
prompt = [
    {
        "role": "system",
        "content": "You are a data annotator specialized in hate speech detection."
    },
    {
        "role": "user",
        "content": """Read the provided TEXT and extract the exact substring that correspond to the following Hate Speech components.
        If a component is not present in the text, you must return the string "absent".

        Below you find the components to identify in the given text:

        - Abuse: Content which involves improper treatment of a person or group, often to unfairly or improperly gain benefit.
        - Age
        - Appearence
        - Disability
        - Disease
        - Ethnicity: Any reference to a person's or group's ethnic origin.
        - Gender: Any reference to a person's or group's gender identity or biological sex.
        - Hate
        - Insult
        - Political Opinion
        - Race
        - Discrimination
        - Religion: Any reference to a person's or group's religious beliefs, practices, or affiliation.
        - Sexual Orientation: Any reference to a person's or group's sexual orientation.
        - Violence
        - Harrassing
        - Insulting
        - Intimidating


        RULES:
        1- You must extract exact quotes from the text. Do not paraphrase or change any words.
        2- Do not output any conversational text, greetings, introductions, or explanations.
        3- Your response must be only the raw json object. Do not wrap the json in markdown formatting.

        OUTPUT FORMAT:
        {
            "Abuse": "extracted substring or absent",
            "Age": "extracted substring or absent",
            "Appearence": "extracted substring or absent",
            "Disability": "extracted substring or absent",
            "Disease": "extracted substring or absent",
            "Ethnicity": "extracted substring or absent",
            "Gender": "extracted substring or absent",
            "Hate": "extracted substring or absent",
            "Insult": "extracted substring or absent",
            "Political Opinion": "extracted substring or absent",
            "Race": "extracted substring or absent",
            "Discrimination": "extracted substring or absent",
            "Religion": "extracted substring or absent",
            "Sexual Orientation": "extracted substring or absent",
            "Violence": "extracted substring or absent",
            "Harrassing": "extracted substring or absent",
            "Insulting": "extracted substring or absent",
            "Intimidating": "extracted substring or absent"
        }


        TEXT: "{text}"

        ANSWER:

        """
    }
]

In [15]:
def prepare_prompts(texts, prompt_template, tokenizer):
  """
    This function format input text samples into instructions prompts.

    Inputs:
      texts: input texts to classify via prompting
      prompt_template: the prompt template provided in this assignment
      tokenizer: the transformers Tokenizer object instance associated
      with the chosen model card

    Outputs:
      input texts to classify in the form of instruction prompts
  """
  formatted_prompts=[]

  for text in texts:
    prompt_copy = [
            {"role": prompt_template[0]["role"], "content": prompt_template[0]["content"]},
            {
                "role": prompt_template[1]["role"],
                "content": prompt_template[1]["content"].replace("{text}", text)
            }
        ]


    formatted_prompt = tokenizer.apply_chat_template(
      prompt_copy,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
    )

    formatted_prompts.append(formatted_prompt)


  return formatted_prompts

# Inference

In [16]:
def generate_responses(model, prompt_examples, tokenizer, verbose=False):
  """
    This function implements the inference loop for a LLM model.
    Given a set of examples, the model is tasked to generate
    a response.

    Inputs:
      model: LLM model instance for prompting
      prompt_examples: pre-processed text samples

    Outputs:
      generated responses
  """
  outputs=[]
  for index, prompt_example in enumerate(prompt_examples):
    if verbose and index % 10 == 0:
      print(f"generating response n.{index}...")
    prompt_example=prompt_example.to(model.device)
    output = model.generate(**prompt_example, max_new_tokens=500,pad_token_id=tokenizer.eos_token_id)
    output = tokenizer.decode(output[0][prompt_example["input_ids"].shape[-1]:], skip_special_tokens=True)
    outputs.append(output)

  return outputs

In [17]:
outputs = generate_responses(model, prepare_prompts(texts, prompt, tokenizer), tokenizer, verbose=True)
#40 minuti

generating response n.0...
generating response n.10...
generating response n.20...
generating response n.30...
generating response n.40...
generating response n.50...
generating response n.60...
generating response n.70...
generating response n.80...
generating response n.90...


In [18]:
parsed_outputs = []
json_errors = []
for output in outputs:
    try:
        parsed_outputs.append(json.loads(output))
    except json.JSONDecodeError:
        json_errors.append(output)

with open("results_evalita_3.json", "w", encoding="utf-8") as f:
    json.dump(parsed_outputs, f, ensure_ascii=False, indent=2)

with open("json_errors_evalita_3.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(json_errors))

# Extraction Results

In [ ]:
def highlights(texts, outputs):
    """
    This function displays the given text with the model extracted highlights.
    """
    colori = {
        "Abuse":              "#FFB3B3",  # rosso chiaro
        "Age":                "#FFE4B3",  # arancio chiaro
        "Appearence":         "#FFF3B3",  # giallo chiaro
        "Disability":         "#B3FFE4",  # verde acqua
        "Disease":            "#B3F0FF",  # azzurro chiaro
        "Ethnicity":          "#B3D9FF",  # blu chiaro
        "Gender":             "#E6B3FF",  # viola chiaro
        "Hate":               "#FF9999",  # rosso
        "Insult":             "#FFB3D9",  # rosa
        "Political Opinion":  "#C8FFB3",  # verde chiaro
        "Race":               "#B3FFFF",  # ciano chiaro
        "Discrimination":     "#FFD9B3",  # pesca
        "Religion":           "#B3FFB3",  # verde
        "Sexual Orientation": "#F0B3FF",  # lilla
        "Violence":           "#FF8080",  # rosso scuro
        "Harrassing":         "#FFB3B3",  # rosso chiaro 2
        "Insulting":          "#FFD4B3",  # albicocca
        "Intimidating":       "#D4B3FF",  # lavanda
    }

    testo_colorato = texts
    for categoria, estratto in outputs.items():
        if estratto and estratto.lower() not in ("null", "absent") and estratto in testo_colorato:
            colore = colori.get(categoria, "#DDDDDD")
            tag_html = (
                f"<mark style='background-color: {colore}; padding: 2px 4px; border-radius: 4px;'>"
                f"{estratto} "
                f"<span style='font-size: 0.7em; font-weight: bold;'>[{categoria}]</span>"
                f"</mark>"
            )
            testo_colorato = testo_colorato.replace(estratto, tag_html)

    display(HTML(f"<p style='line-height: 1.8; font-size: 16px;'>{testo_colorato}</p>"))


In [ ]:
for i in range(len(outputs)):
    parsed_output = {}
    try:
        parsed_output = json.loads(outputs[i])
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e} for text: {texts[i]}, model output: {outputs1[i]}")

    highlights(texts[i], parsed_output)

JSONDecodeError: Unterminated string starting at: line 15 column 5 (char 743) for text: BIBI non si è fatto condizionare dai comunisti di merda del mondo, Obama, Europa, ecc. Ha VINTO e ora avanti con l'annullamento dell'islam, model output: {
    "Abuse": "del mondo, del mondo",
    "Age": "absent",
    "Appearence": "absent",
    "Disability": "absent",
    "Disease": "absent",
    "Ethnicity": "del mondo",
    "Gender": "absent",
    "Hate": "BIBI non si è fatto condizionare dai comunisti di merda del mondo, Obama, Europa, ecc. Ha VINTO e ora avanti con l'annullamento dell'islam",
    "Insult": "di merda, Obama, Europa, ecc.",
    "Political Opinion": "BIBI non si è fatto condizionare dai comunisti, Obama, Europa, ecc. Ha VINTO e ora avanti con l'annullamento dell'islam",
    "Race": "del mondo",
    "Discrimination": "BIBI non si è fatto condizionare dai comunisti, Obama, Europa, ecc. Ha VINTO e ora avanti con l'annullamento dell'islam",
    "Religion": "islam",
    "Sexual


JSONDecodeError: Expecting ':' delimiter: line 11 column 78 (char 298) for text: #Palermo. Allaccio non regolare al campo rom, dopo 4 mesi non è cambiato nulla: https://t.co/9lP58YEkSN @LeolucaOrlando1, model output: {
    "Abuse": "absent",
    "Age": "absent",
    "Appearence": "absent",
    "Disability": "absent",
    "Disease": "absent",
    "Ethnicity": "rom",
    "Gender": "absent",
    "Hate": "absent",
    "Insult": "absent",
    "Political Opinion": "Allaccio non regolare al campo", "LeolucaOrlando1" is likely a political figure's name, 
    "Race": "rom",
    "Discrimination": "absent",
    "Religion": "absent",
    "Sexual Orientation": "absent",
    "Violence": "absent",
    "Harrassing": "absent",
    "Insulting": "absent",
    "Intimidating": "absent"
}


JSONDecodeError: Unterminated string starting at: line 18 column 18 (char 695) for text: @Cartabiancarai3 @KhalidChaouki3 altro che islam o cristianesimo!!! ...a c dobbiamo sorbire anche "dell'ignorante" : islam è  uno solo!!, model output: {
    "Abuse": "altro che islam o cristianesimo!!! ...a c dobbiamo sorbire anche ",
    "Age": "absent",
    "Appearence": "absent",
    "Disability": "absent",
    "Disease": "absent",
    "Ethnicity": "islam o cristianesimo!!!",
    "Gender": "absent",
    "Hate": "islam o cristianesimo!!!",
    "Insult": "altro che islam o cristianesimo!!! ...a c dobbiamo sorbire anche ",
    "Political Opinion": "absent",
    "Race": "islam o cristianesimo!!!",
    "Discrimination": "altro che islam o cristianesimo!!! ...a c dobbiamo sorbire anche ",
    "Religion": "islam o cristianesimo!!!",
    "Sexual Orientation": "absent",
    "Violence": "absent",
    "Harrassing": "absent",
    "Insulting": "altro che islam o cristianesimo!!! ...a c dobb
